# DFU-ImageGuard — Complete Q1-Oriented Experiment

Run the single executable cell below. It performs the locked leakage-safe five-fold experiment, saves exactly five primary checkpoints, and then applies Q1 post-hoc corrections without retraining: threshold-safe fold metrics, duplicate-group confidence intervals, cluster-aware comparisons, threshold-aware uncertainty, and automatic claim safeguards. No output is prefilled.


In [ ]:
import os, sys, subprocess, shutil, importlib
from pathlib import Path
PACKAGES=["timm>=1.0.9","numpy>=1.26","pandas>=2.2","scikit-learn>=1.5","scipy>=1.13","matplotlib>=3.9","Pillow>=10.4","ImageHash>=4.3","kagglehub>=0.3","joblib>=1.4","shap>=0.46","lime>=0.2.0.1","grad-cam>=1.5","opencv-python-headless>=4.10","pyyaml>=6.0","tabulate>=0.9"]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--disable-pip-version-check",*PACKAGES])
REPO=Path("/content/DFU-ImageGuard")
if (REPO/".git").exists():
    subprocess.run(["git","-C",str(REPO),"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",str(REPO),"reset","--hard","origin/main"],check=True)
else:
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.check_call(["git","clone","https://github.com/AzizulHakim00/DFU-ImageGuard.git",str(REPO)])
for module_name in list(sys.modules):
    if module_name=="src" or module_name.startswith("src."): del sys.modules[module_name]
importlib.invalidate_caches()
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
try:
    from google.colab import userdata
    for secret_name in ["GITHUB_TOKEN","HF_TOKEN"]:
        try:
            value=userdata.get(secret_name)
            if value: os.environ[secret_name]=value
        except Exception: pass
except Exception: pass
commit=subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"],check=True,capture_output=True,text=True).stdout.strip()
print(f"DFU-ImageGuard repository commit: {commit}")
from src.config_data import storage_write_probe
from src.q1_pipeline import run_complete_pipeline
try:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
    print("Storage preflight:",storage_write_probe(Path("/content/drive/MyDrive/DFU-ImageGuard")))
except Exception as exc: print(f"Drive preflight note: {exc}")
OVERRIDES={"LOCAL_REPO":str(REPO),"FORCE_RETRAIN":False,"RUN_BASELINES":True,"RUN_ROBUSTNESS":True,"RUN_XAI":True,"N_FOLDS":5,"MAX_EPOCHS":30,"PATIENCE":7,"BOOTSTRAP_REPS":1000,"NUM_WORKERS":2}
FINAL_REPORT=run_complete_pipeline(OVERRIDES)
FINAL_REPORT
